<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 1 — Biyoaktivite Verisinin Hazırlanması

> **Kurs iş akışındaki yeri:** Bu, altı not defterinden **ilkidir**. Burada üretilen
> iki dosya (`ntrk1_regresyon.csv` ve `ntrk1_siniflandirma.csv`) sonraki bütün
> oturumların girdisidir.

Makine öğrenmesi tabanlı QSAR çalışmalarında harcanan emeğin büyük bölümü model
kurmaya değil, **veriyi modellenebilir hâle getirmeye** gider. Bu not defteri o
işi uçtan uca yapar: ChEMBL veritabanındaki ham, dağınık ve tekrarlı deneysel
ölçümleri, bir modelin öğrenebileceği temiz bir **"molekül → aktivite"** tablosuna
dönüştürür.

Sonunda iki veri seti üretilecektir:

| Veri seti | Hedef değişken | Kullanılacağı oturum |
|---|---|---|
| **Regresyon** | `pIC50` (sürekli sayı) | Oturum 3 |
| **Sınıflandırma** | `Aktiflik` (0 / 1) | Oturum 4 |

---

## Hedef protein: NTRK1 (TrkA) — yüksek afiniteli sinir büyüme faktörü reseptörü

| | |
|---|---|
| **Gen** | `NTRK1` |
| **UniProt** | [`P04629`](https://www.uniprot.org/uniprotkb/P04629) |
| **ChEMBL hedef** | [`CHEMBL2815`](https://www.ebi.ac.uk/chembl/target_report_card/CHEMBL2815/) |
| **Protein sınıfı** | Reseptör tirozin kinaz (RTK) |
| **Doğal ligandı** | Sinir büyüme faktörü (NGF) |

**Neden bu hedef?** Üç gerekçeyle seçilmiştir:

1. **Klinik olarak doğrulanmış ve güncel.** `NTRK` gen füzyonlarını hedefleyen
   **larotrektinib** ve **entrektinib**, FDA tarihinde ilk kez *tümörün bulunduğu
   dokuya değil, taşıdığı genetik değişime* göre onaylanan ilaçlar oldu
   (tümör-agnostik onay). Yani bu hedef, moleküler belirteç temelli ilaç
   tasarımının ders kitabı örneğidir.
2. **Veri açısından zengin.** ChEMBL'de bu hedef için **10.600'den fazla**
   IC50 kaydı bulunur; bunların **8.400 kadarı** kesin (`=`) ilişkili ve `nM`
   birimli ölçümdür. Kalanı sansürlüdür (`>`, `<`) ve Bölüm 7'de göreceğiniz
   gibi regresyona katılmaz.
3. **Direnç sorunu gerçek.** Birinci nesil inhibitörlere karşı gelişen direnç
   mutasyonları (ör. G595R, G667C) ikinci nesil ilaçların (selitrektinib,
   repotrektinib) geliştirilmesini gerektirdi. Bu, "aktiflik tahmini"nin neden
   sürekli yenilenen bir problem olduğunu gösterir.

> **Not.** NTRK1, ağrı biyolojisinde de merkezi bir rol oynar (NGF–TrkA yolağı);
> bu nedenle onkoloji dışında analjezik geliştirme çalışmalarının da hedefidir.

---

## Bu not defterindeki iş akışı

```
ChEMBL (canlı sorgu) ──┐
                       ├──► karşılaştırma ──► temizlik hattı ──► pIC50 ──┬──► regresyon veri seti
Depodaki hazır CSV  ──┘                                                   └──► sınıflandırma veri seti
```

Veriyi **iki yoldan da** alacağız:

- **Yol A — canlı ChEMBL sorgusu:** Gerçek araştırma pratiği. Veritabanı
  günceldir, ama ağ hatalarına ve veritabanı sürümüne bağlıdır.
- **Yol B — depodaki hazır CSV:** Kurs deposunda dondurulmuş, bilinen bir sürüm.
  Tekrarlanabilirliği garanti eder.

Ardından ikisini **karşılaştıracağız**. Bu karşılaştırma yapay bir alıştırma
değildir: aynı sorgunun farklı zamanlarda farklı sonuç vermesi, hesaplamalı
çalışmalarda **veri sürümü bildirmenin** neden zorunlu olduğunu gösterir.

---
## 1. Yapılandırma

Not defteri boyunca kullanılacak bütün sabitler bu tek hücrede toplanmıştır.
Başka bir hedefle çalışmak isterseniz **yalnızca bu hücreyi** değiştirmeniz
yeterlidir; geri kalan kod hedefe özel hiçbir varsayım yapmaz.

İki ayarı özellikle not edin:

- **`VERI_KAYNAGI`** — hangi yolun kullanılacağını belirler. Kursta varsayılan
  olarak `"ikisi"` ile çalışacağız: her iki yol da çalıştırılır ve karşılaştırılır.
- **`AKTIFLIK_ESIGI`** — sınıflandırmada bir molekülü "aktif" saymak için
  gereken en düşük `pIC50` değeri. Bu sayı **bilimsel bir karardır**,
  teknik bir detay değil; Bölüm 13'te ayrıntılı tartışacağız.

In [ ]:
# ── Hedef ─────────────────────────────────────────────────────────────────────
TARGET_CHEMBL_ID = "CHEMBL2815"      # NTRK1 / TrkA  (UniProt: P04629)
TARGET_ADI       = "NTRK1"
AKTIVITE_TIPI    = "IC50"       # Modellenecek olcum tipi
HEDEF_KOLON      = "pIC50"    # -log10(IC50 [M])

# ── Siniflandirma esigi ───────────────────────────────────────────────────────
AKTIFLIK_ESIGI = 7.5               # pIC50 >= 7.5  ->  Aktif (IC50 <= 32 nM)

# ── Veri kaynagi: "ikisi" | "chembl" | "depo" ─────────────────────────────────
VERI_KAYNAGI = "ikisi"

# ── Google Drive proje klasoru ────────────────────────────────────────────────
# Butun oturumlar (01-06) bu AYNI klasoru paylasir: bir defterin ciktisi
# bir sonrakinin girdisidir.
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"   # Colab disi ortam icin

# ── Kurs deposu (hazir veri buradan okunur) ───────────────────────────────────
DEPO_RAW = "https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main"

# ── Cikti dosya adlari ────────────────────────────────────────────────────────
AD_HAM = "ntrk1_ham_chembl.csv"
AD_REG = "ntrk1_regresyon.csv"
AD_SNF = "ntrk1_siniflandirma.csv"

print(f"Hedef          : {TARGET_ADI} ({TARGET_CHEMBL_ID})")
print(f"Aktivite tipi  : {AKTIVITE_TIPI}  ->  {HEDEF_KOLON}")
print(f"Aktiflik esigi : {HEDEF_KOLON} >= {AKTIFLIK_ESIGI}")
print(f"Veri kaynagi   : {VERI_KAYNAGI}")

---
## 2. Google Drive'ı bağlama

Colab oturumları **geçicidir**: çalışma zamanı kapandığında `/content/` altındaki
her şey silinir. Ürettiğimiz veri setlerinin kalıcı olması ve sonraki
oturumların onlara erişebilmesi için çıktıları **Google Drive**'a yazacağız.

Hücre çalıştığında Colab, Drive erişimi için **izin isteyecektir**: açılan
pencereden Google hesabınızı seçip onay vermeniz gerekir.

> Kod, `google.colab` modülünü bulamazsa (yani Colab dışında, kendi
> bilgisayarınızda çalıştırıyorsanız) otomatik olarak yerel bir klasöre düşer.
> Böylece not defteri her iki ortamda da çalışır.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE = Path(DRIVE_PROJE_KLASORU)
    ORTAM = "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE = Path(YEREL_KLASOR)
    ORTAM = "Yerel"

PROJE.mkdir(parents=True, exist_ok=True)

# Tam cikti yollari
YOL_HAM = PROJE / AD_HAM
YOL_REG = PROJE / AD_REG
YOL_SNF = PROJE / AD_SNF

print(f"Ortam          : {ORTAM}")
print(f"Proje klasoru  : {PROJE}")

---
## 3. Kütüphanelerin kurulumu

İki kütüphaneye ihtiyacımız var:

- **`rdkit`** — kemoinformatiğin standart açık kaynak kütüphanesi. Burada
  SMILES dizgilerini doğrulamak ve **kanonik** (tek, standart) biçime çevirmek
  için kullanacağız.
- **`requests`** — ChEMBL'in web servisine HTTP istekleri göndermek için
  (Colab'da hazır gelir).

Kurulum oturum başına **bir kez** gerekir; `-q` çıktıyı sessize alır.

> **ChEMBL istemcisi hakkında.** ChEMBL'in resmî `chembl_webresource_client`
> paketi de vardır. Bu kursta bunun yerine REST arayüzüne **doğrudan** istek
> atacağız; böylece neyin istendiğini ve hata durumunda ne yapılacağını
> tamamen kendimiz denetleyeceğiz (Bölüm 5).

In [ ]:
!pip install rdkit -q
print("Kurulum tamam.")

---
## 4. Kütüphanelerin içe aktarılması

In [ ]:
import json
import math
import time
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")        # RDKit'in ayristirma uyarilarini sustur

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.dpi"] = 110

print("Kutuphaneler yuklendi.")

---
## 5. ChEMBL'e dayanıklı sorgu atmak

ChEMBL web servisi, büyük sorgularda zaman zaman **geçici `500` hataları**
döndürür: aynı istek bir denemede başarısız olurken bir sonrakinde çalışabilir.
Kurs hazırlığı sırasında bu davranış doğrudan gözlendi.

Bu, gerçek bir araştırma pratiği sorunudur ve çözümü basittir: isteği
**artan bekleme süreleriyle yeniden denemek** (exponential backoff). Aşağıdaki
`chembl_sorgu` fonksiyonu bunu yapar — başarısız olursa bekler, bekleme süresini
katlayarak artırır ve yeniden dener.

> ⚠️ **Kurs hazırlığında gözlenen durum.** Bu materyal hazırlanırken ChEMBL
> servisinin isteklerin yaklaşık **yarısını** reddettiği bir dönem yaşandı —
> sorgunun biçiminden bağımsız olarak, rastgele. Böyle dönemlerde yeniden
> deneme bile yetmeyebilir. Bu yüzden Bölüm 9'daki **depo verisi** yolu
> yalnızca bir kolaylık değil, oturumun sigortasıdır.

> **Niçin önemli?** 15 kişilik bir sınıf aynı anda sorgu attığında bu hatalarla
> karşılaşma olasılığı yükselir. Yeniden deneme mantığı olmayan bir kod,
> tamamen sağlam bir veri hattını "çalışmıyor" gibi gösterir.

`chembl_aktiviteleri_indir` fonksiyonu ise **sayfalama** (pagination) yapar:
ChEMBL bir istekte en çok 1000 kayıt döndürdüğü için, `offset` değerini
artırarak bütün sayfaları sırayla toplar.

In [ ]:
CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data"


def chembl_sorgu(uc_nokta: str, parametreler: dict, deneme: int = 15) -> dict:
    """ChEMBL REST istegi; gecici 500 hatalarina karsi artan beklemeyle yeniden dener."""
    url = f"{CHEMBL_API}/{uc_nokta}.json"
    bekleme, son_hata = 1.0, None
    for k in range(deneme):
        try:
            yanit = requests.get(url, params=parametreler, timeout=120)
            yanit.raise_for_status()
            return yanit.json()
        except Exception as hata:
            son_hata = hata
            if k < deneme - 1:
                time.sleep(bekleme)
                bekleme = min(bekleme * 1.8, 30)   # 1s, 1.8s, 3.2s, ... en cok 30s
    raise RuntimeError(f"ChEMBL {deneme} denemede yanit vermedi: {son_hata}")


def chembl_kayit_sayisi(target_id: str, tip: str) -> int:
    """Bir aktivite tipi icin kayit sayisi (veriyi indirmeden, tek hafif istek)."""
    yanit = chembl_sorgu("activity", {
        "target_chembl_id": target_id,
        "standard_type": tip,
        "standard_relation": "=",      # yalnizca kesin olcumler
        "standard_units": "nM",        # birim tutarliligi
        "limit": 1,
    })
    return yanit["page_meta"]["total_count"]


def chembl_aktiviteleri_indir(target_id: str, tip: str) -> pd.DataFrame:
    """Hedefin verilen tipteki butun kesin (=) ve nM kayitlarini sayfalayarak indirir."""
    kayitlar, offset = [], 0
    while True:
        yanit = chembl_sorgu("activity", {
            "target_chembl_id": target_id,
            "standard_type": tip,
            "standard_relation": "=",
            "standard_units": "nM",
            "limit": 1000,
            "offset": offset,
            "only": ("molecule_chembl_id,canonical_smiles,standard_type,"
                     "standard_relation,standard_value,standard_units,"
                     "assay_chembl_id,pchembl_value"),
        })
        sayfa = yanit.get("activities", [])
        kayitlar.extend(sayfa)
        toplam = yanit["page_meta"]["total_count"]
        print(f"  indirildi: {len(kayitlar):>6} / {toplam}")
        if len(sayfa) < 1000:
            break
        offset += 1000
    return pd.DataFrame(kayitlar)


print("Sorgu fonksiyonlari hazir.")

---
## 6. Hangi ölçüm tipini modellemeliyiz?

Bir QSAR modeli **tek ve tutarlı bir ölçüm tipi** üzerine kurulmalıdır. Farklı
deney tiplerini (ör. IC50 ile Ki) aynı torbaya koymak modele gürültü katar,
çünkü bunlar farklı deneysel düzeneklerden gelir ve farklı şeyleri ölçer:

| Ölçüm | Ne ölçer? |
|---|---|
| **IC50** | Hedefin aktivitesini **%50 baskılamak** için gereken derişim (fonksiyonel) |
| **Ki** | Bağlanma afinitesinin denge sabiti (termodinamik) |
| **Kd** | Ayrışma sabiti (doğrudan bağlanma) |
| **EC50** | %50 **etki** oluşturan derişim (hücresel/fonksiyonel) |

Aşağıdaki hücre, her tip için kaç **kesin** (`=`) ve **nM** birimli kayıt
olduğunu sayar. Dikkat edin: veriyi indirmiyoruz, yalnızca sayıyoruz — bu yüzden
saniyeler sürer.

Hedefimiz için en çok veri **IC50**'de olduğu için modellemeyi onun üzerine
kuracağız.

In [ ]:
TIPLER = ["IC50", "Ki", "Kd", "EC50", "Potency", "Inhibition"]

sayimlar = []
for tip in TIPLER:
    # Her tip BAGIMSIZ olarak denenir: biri basarisiz olursa digerleri yine
    # sayilir ve hucre cokmez.
    try:
        n = chembl_kayit_sayisi(TARGET_CHEMBL_ID, tip)
        print(f"  {tip:<12} {n:>7}")
    except Exception as hata:
        n = None
        print(f"  {tip:<12} {'?':>7}   (sunucu yanit vermedi)")
    sayimlar.append({"Aktivite tipi": tip, "Kesin (=) ve nM kayit": n})

df_tipler = (pd.DataFrame(sayimlar)
             .sort_values("Kesin (=) ve nM kayit", ascending=False,
                          na_position="last")
             .reset_index(drop=True))

if df_tipler["Kesin (=) ve nM kayit"].isna().all():
    print("\nHicbir sayim alinamadi. ChEMBL servisi su anda yanit vermiyor;")
    print("Bolum 9'daki depo verisiyle devam edebilirsiniz.")
else:
    print(f"\nModellenecek tip: {AKTIVITE_TIPI}")
df_tipler

---
## 7. Yol A — ChEMBL'den canlı veri çekme

Şimdi seçtiğimiz tipteki bütün kayıtları indiriyoruz. Sorguya üç filtre
koyuyoruz ve her birinin **bilimsel bir gerekçesi** var:

| Filtre | Gerekçe |
|---|---|
| `standard_type = IC50` | Tek, tutarlı bir ölçüm tipi (Bölüm 6) |
| `standard_relation = "="` | **Kesin** ölçümler. `>` veya `<` içeren kayıtlar "sansürlü" (censored) değerlerdir: "IC50 > 10000 nM" demek, gerçek değerin bilinmediği anlamına gelir. Bunları regresyona gerçek sayı gibi vermek modeli yanıltır. |
| `standard_units = "nM"` | pIC50 hesabı molar derişim gerektirir. ChEMBL'de değerler bazen `µg.mL⁻¹` gibi birimlerde gelir; bunlar mol kütlesi bilinmeden çevrilemez. |

> ⏱️ **Süre:** Binlerce kayıt sayfa sayfa inecektir; bu hücre **birkaç dakika**
> sürebilir. İndirme bittikten sonra ham veriyi hemen Drive'a kaydediyoruz ki
> hücreyi tekrar çalıştırmak zorunda kalmayın.

In [ ]:
ham_chembl = None

if VERI_KAYNAGI in ("ikisi", "chembl"):
    print(f"'{TARGET_CHEMBL_ID}' icin {AKTIVITE_TIPI} kayitlari indiriliyor...")
    try:
        ham_chembl = chembl_aktiviteleri_indir(TARGET_CHEMBL_ID, AKTIVITE_TIPI)
        ham_chembl.to_csv(YOL_HAM, index=False)
        print(f"\nHam kayit sayisi : {len(ham_chembl)}")
        print(f"Kolon sayisi     : {ham_chembl.shape[1]}")
        print(f"Kaydedildi       : {YOL_HAM}")
    except Exception as hata:
        print(f"\n!! Canli cekim basarisiz oldu: {hata}")
        print("   Sorun degil: Yol B (depodaki hazir veri) ile devam edebiliriz.")
        ham_chembl = None
else:
    print("VERI_KAYNAGI='depo' secildi; canli cekim atlandi.")

if ham_chembl is not None:
    display(ham_chembl.head())

---
## 8. Temizlik hattı

Ham deneysel veri her zaman "kirlidir": eksik değerler, aynı molekülün birden
çok ölçümü, kimyasal olarak geçersiz yapılar ve aynı molekülün farklı SMILES
yazımları içerir. Aşağıdaki fonksiyon temizliği **altı adımda** yapar ve her
adımda kaç molekül kaldığını kaydeder.

| Adım | Ne yapar? | Neden? |
|---|---|---|
| 1 | Eksik kimlik / değer / SMILES satırlarını atar | Modellenemez kayıtlar |
| 2 | Değerleri sayıya çevirir, pozitif olanları tutar | Logaritma için `> 0` zorunlu |
| 3 | Molekül başına **en düşük** değeri seçer | Aynı molekülün birden çok ölçümü varsa tek temsilci gerekir; en düşük IC50 = en potent ölçüm |
| 4 | SMILES'ı **kanonikleştirir** | Aynı yapı farklı yazılabilir (`OCC` ve `CCO` ikisi de etanol). Kanonikleştirme tek standart biçim üretir. |
| 5 | Kanonik yapıya göre **yeniden** tekilleştirir | Adım 4'ten sonra ortaya çıkan "gizli tekrarları" çözer |
| 6 | `pIC50` ve `Aktiflik` kolonlarını hesaplar | Model hedefleri |

**Adım 4–5 neden kritik?** Farklı SMILES yazımları yüzünden aynı molekül veri
setinde iki ayrı satır gibi görünebilir. Bu "gizli tekrarlar" yalnızca veriyi
şişirmez; eğitim/test bölünmesinde **aynı molekülün hem eğitimde hem testte
yer almasına** yol açar. Buna **veri sızıntısı** (data leakage) denir ve modelin
performansını olduğundan iyi gösterir.

In [ ]:
def kanonik_smiles(smiles):
    """SMILES'i kanonik RDKit gosterimine cevirir; gecersiz/eksikse None."""
    if not isinstance(smiles, str) or not smiles:
        return None
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None


def temizle(ham: pd.DataFrame, tip: str, esik: float):
    """Ham ChEMBL kayitlarini modellenebilir tabloya cevirir.

    Geri dondurur: (temiz DataFrame, huni listesi)
    """
    hedef = f"p{tip}"
    huni = [("Ham kayitlar", len(ham))]

    # 1) Eksik kimlik / deger / SMILES
    df = ham.dropna(subset=["molecule_chembl_id", "standard_value",
                            "canonical_smiles"]).copy()
    huni.append(("Kimlik + deger + SMILES tam", len(df)))

    # 2) Sayisala cevir, pozitif olanlari tut
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
    df = df.dropna(subset=["standard_value"])
    df = df[df["standard_value"] > 0]
    huni.append(("Sayisal ve pozitif deger", len(df)))

    # 3) Molekul basina en potent (en kucuk) olcum
    df = df.loc[df.groupby("molecule_chembl_id")["standard_value"].idxmin()].copy()
    huni.append(("Molekul basina min deger", len(df)))

    # 4) Kanoniklestirme (gecersiz yapilar burada elenir)
    df["Smiles"] = df["canonical_smiles"].apply(kanonik_smiles)
    gecersiz = int(df["Smiles"].isna().sum())
    df = df.dropna(subset=["Smiles"])
    huni.append((f"Gecerli kanonik SMILES (elenen: {gecersiz})", len(df)))

    # 5) Kanonik yapiya gore yeniden tekillestir
    df = df.loc[df.groupby("Smiles")["standard_value"].idxmin()].copy()
    huni.append(("Kanonik yapiya gore benzersiz", len(df)))

    # 6) Hedef degiskenler:  pIC50 = -log10(IC50[nM] x 1e-9)
    df[hedef] = -np.log10(df["standard_value"] * 1e-9)
    df["Aktiflik"] = (df[hedef] >= esik).astype(int)

    df = (df[["molecule_chembl_id", "Smiles", "standard_value", hedef, "Aktiflik"]]
          .rename(columns={"standard_value": f"{tip}_nM"})
          .sort_values(hedef, ascending=False)
          .reset_index(drop=True))
    huni.append(("NIHAI veri seti", len(df)))
    return df, huni


print("Temizlik fonksiyonlari hazir.")

### Temizliği uygulama

Şimdi hattı canlı indirilen veriye uygulayıp **daralma hunisini** (filtering
funnel) yazdıralım. Bu tablo iki açıdan değerlidir:

- **Bilimsel raporlama:** Bir makalenin "Yöntem" bölümündeki veri küratörlüğü
  adımlarını doğrudan bu tabloyla sunabilirsiniz.
- **Hata ayıklama:** Bir adımda beklenmedik büyüklükte kayıp olursa (ör. verinin
  %90'ı gidiyorsa) sorun hemen görünür hâle gelir.

In [ ]:
df_canli, huni_canli = (temizle(ham_chembl, AKTIVITE_TIPI, AKTIFLIK_ESIGI)
                        if ham_chembl is not None else (None, None))

if df_canli is not None:
    print("Temizlik hunisi (canli ChEMBL verisi)\n" + "-" * 52)
    for ad, n in huni_canli:
        print(f"  {ad:<40} {n:>6}")
    print(f"\n{HEDEF_KOLON} araligi: {df_canli[HEDEF_KOLON].min():.2f} - "
          f"{df_canli[HEDEF_KOLON].max():.2f}")
    display(df_canli.head())
else:
    print("Canli veri yok; Yol B ile devam.")

---
## 9. Yol B — depodaki hazır veri seti

Kurs deposunda, yukarıdaki **tam aynı** hattan geçirilmiş dondurulmuş bir sürüm
bulunur. Bunu üreten betiği de depoda bulabilirsiniz:
[`araclar/veri_uret.py`](https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main/araclar/veri_uret.py).

İki gerekçeyle bu dosyaya ihtiyaç var:

1. **Tekrarlanabilirlik.** ChEMBL sürekli güncellenir. Bir makaleyi yıllar sonra
   tekrar üretmek isteyen biri "ChEMBL'den indirdim" ifadesiyle aynı sonuca
   ulaşamaz; veri setinin dondurulmuş hâline ihtiyaç duyar.
2. **Dayanıklılık.** Derste ağ ya da sunucu kaynaklı bir sorun olursa oturum
   kesintiye uğramaz.

Dosyanın yanında bir de **künye** (`ntrk1_veri_kunyesi.json`) vardır: hangi tarihte, hangi
hedef ve eşikle üretildiğini kaydeder. Veri setlerinize böyle bir künye eklemek
iyi bir alışkanlıktır.

In [ ]:
df_depo = None

if VERI_KAYNAGI in ("ikisi", "depo"):
    url_snf   = f"{DEPO_RAW}/veri/ntrk1_siniflandirma.csv"
    url_kunye = f"{DEPO_RAW}/veri/ntrk1_veri_kunyesi.json"
    print(f"Indiriliyor: {url_snf}")
    try:
        df_depo = pd.read_csv(url_snf)
        print(f"Depo veri seti sekli: {df_depo.shape}")

        kunye = requests.get(url_kunye, timeout=60).json()
        print("\nVeri kunyesi")
        print("-" * 46)
        for anahtar in ("hedef_adi", "hedef_chembl_id", "aktivite_tipi",
                        "siniflandirma_esigi", "uretim_tarihi",
                        "nihai_molekul", "aktif", "inaktif", "aktif_yuzde"):
            if anahtar in kunye:
                print(f"  {anahtar:<22}: {kunye[anahtar]}")
    except Exception as hata:
        print(f"!! Depo verisi okunamadi: {hata}")
        print("   (Depo henuz yayinlanmamis olabilir; Yol A ile devam edin.)")
        df_depo = None
else:
    print("VERI_KAYNAGI='chembl' secildi; depo verisi atlandi.")

if df_depo is not None:
    display(df_depo.head())

---
## 10. İki yolu karşılaştırma

Şimdi işin öğretici kısmı: **aynı sorgu, aynı temizlik hattı — sonuç aynı mı?**

Üç şeye bakacağız:

1. **Molekül sayısı** — kaç molekül var, fark ne kadar?
2. **Yapı kesişimi** — kanonik SMILES kümeleri ne kadar örtüşüyor?
3. **Ortak moleküllerde değer uyumu** — aynı molekül için iki kaynakta aynı
   `pIC50` var mı?

**Ne beklemeliyiz?** Küçük farklar **normaldir ve beklenir**. Depo verisi belirli
bir tarihte dondurulmuştur; canlı ChEMBL o tarihten sonra yeni ölçümler eklemiş,
bazı kayıtları düzeltmiş ya da geri çekmiş olabilir. Fark büyükse bu da bilgidir:
veritabanı sürümünün sonuçları ne kadar etkileyebileceğini gösterir.

> **Araştırma pratiği.** Yayınlarınızda şu üçünü mutlaka bildirin: ChEMBL
> **sürümü**, indirme **tarihi** ve uygulanan **filtreler**. Aksi hâlde
> çalışmanız tekrar üretilemez.

In [ ]:
if df_canli is not None and df_depo is not None:
    set_canli = set(df_canli["Smiles"])
    set_depo  = set(df_depo["Smiles"])
    ortak     = set_canli & set_depo

    print("Molekul sayilari")
    print("-" * 52)
    print(f"  Canli ChEMBL          : {len(set_canli):>6}")
    print(f"  Depo (dondurulmus)    : {len(set_depo):>6}")
    print(f"  Ortak yapi            : {len(ortak):>6}")
    print(f"  Yalnizca canlida      : {len(set_canli - set_depo):>6}")
    print(f"  Yalnizca depoda       : {len(set_depo - set_canli):>6}")
    ortusme = 100 * len(ortak) / len(set_canli | set_depo)
    print(f"  Jaccard ortusmesi     : %{ortusme:.1f}")

    # Ortak molekullerde deger uyumu
    birlesik = (df_canli[["Smiles", HEDEF_KOLON]]
                .merge(df_depo[["Smiles", HEDEF_KOLON]], on="Smiles",
                       suffixes=("_canli", "_depo")))
    fark = (birlesik[f"{HEDEF_KOLON}_canli"] - birlesik[f"{HEDEF_KOLON}_depo"]).abs()

    print("\nOrtak molekullerde deger uyumu")
    print("-" * 52)
    print(f"  Karsilastirilan molekul : {len(birlesik):>6}")
    print(f"  Birebir ayni deger      : {int((fark < 1e-9).sum()):>6}"
          f"  (%{100 * (fark < 1e-9).mean():.1f})")
    print(f"  Ortalama |fark|         : {fark.mean():.4f}")
    print(f"  En buyuk |fark|         : {fark.max():.4f}")
    print(f"  Pearson r               : {birlesik[f'{HEDEF_KOLON}_canli'].corr(birlesik[f'{HEDEF_KOLON}_depo']):.6f}")

    if fark.max() < 1e-9 and len(set_canli ^ set_depo) == 0:
        print("\n=> Iki kaynak TAMAMEN ayni. Veri hatti tekrar uretilebilir.")
    else:
        print("\n=> Kucuk farklar var: ChEMBL depo tarihinden sonra guncellenmis.")
        print("   Bu beklenen bir durumdur; veri surumu bildirmenin neden")
        print("   zorunlu oldugunu gosterir.")
else:
    print("Karsilastirma icin iki kaynak da gerekli.")
    print(f"  Canli veri : {'var' if df_canli is not None else 'YOK'}")
    print(f"  Depo verisi: {'var' if df_depo is not None else 'YOK'}")

### Hangi veriyle devam edeceğiz?

Aşağıdaki hücre, elimizdeki kaynaklardan birini seçer. Kursta oturumun geri
kalanının herkeste **birebir aynı** ilerlemesi için **depo verisini**
tercih ediyoruz; yoksa canlı veriye düşer.

Kendi araştırmanızda tersini yapmak isteyeceksiniz: en güncel veriyi kullanın,
ama kullandığınız sürümü dondurup saklayın.

In [ ]:
if df_depo is not None:
    df = df_depo.copy()
    KAYNAK = "depo (dondurulmus surum)"
elif df_canli is not None:
    df = df_canli[["Smiles", HEDEF_KOLON, "Aktiflik"]].copy()
    KAYNAK = "canli ChEMBL"
else:
    raise RuntimeError(
        "Hic veri yok. Yukaridaki hucrelerden en az birinin basariyla "
        "calismasi gerekir. VERI_KAYNAGI ayarini kontrol edin."
    )

# Esigi burada yeniden uygula: depo dosyasi farkli bir esikle uretilmis olabilir.
df["Aktiflik"] = (df[HEDEF_KOLON] >= AKTIFLIK_ESIGI).astype(int)

print(f"Devam edilen kaynak : {KAYNAK}")
print(f"Molekul sayisi      : {len(df)}")
print(f"Kolonlar            : {list(df.columns)}")
df.head()

---
## 11. Veri setini tanımak

Modelleme öncesi her zaman veriye **bakılır**. Aşağıdaki özet istatistikler ve
sonraki bölümdeki grafikler, modelin öğrenmesi beklenen sinyalin nasıl
dağıldığını gösterir.

`pIC50` değerlerinin yorumu:

| pIC50 | IC50 | Yorum |
|---|---|---|
| 9 | 1 nM | Çok potent |
| 8 | 10 nM | Potent |
| 7 | 100 nM | İyi |
| 6 | 1 µM | Orta |
| 5 | 10 µM | Zayıf |

**Düşük IC50 = güçlü molekül**, dolayısıyla **yüksek pIC50 = güçlü molekül**.
Logaritmik dönüşüm iki işi birden yapar: yönü sezgisel hâle getirir ve çok geniş
bir derişim aralığını (nanomolardan mikromolara) modelin öğrenmesine elverişli,
daha simetrik bir ölçeğe taşır.

In [ ]:
print(f"{HEDEF_KOLON} ozet istatistikleri")
print("-" * 40)
print(df[HEDEF_KOLON].describe().to_string())

print(f"\nEn potent molekul : {HEDEF_KOLON} = {df[HEDEF_KOLON].max():.2f}"
      f"  ({AKTIVITE_TIPI} ~ {10 ** (9 - df[HEDEF_KOLON].max()):.3f} nM)")
print(f"En zayif molekul  : {HEDEF_KOLON} = {df[HEDEF_KOLON].min():.2f}"
      f"  ({AKTIVITE_TIPI} ~ {10 ** (9 - df[HEDEF_KOLON].min()):,.0f} nM)")

---
## 12. Sınıflandırma eşiği: bilimsel bir karar

Sınıflandırma modeli için molekülleri **aktif** ve **inaktif** diye ikiye
ayırmamız gerekir. Bu ayrımın eşiği keyfi bir sayı değildir; iki şeyi birlikte
düşünmek gerekir:

**1. Farmakolojik anlam.** Kinaz inhibitörü geliştirme çalışmalarında
`pIC50 = 7` (yani IC50 ≤ 100 nM) yaygın bir "ilaç benzeri potens"
sınırıdır. Daha gevşek bir eşik (6, yani 1 µM) zayıf bağlanıcıları da aktif
sayar; daha katı bir eşik (8, yani 10 nM) yalnızca optimize edilmiş bileşikleri
alır.

**2. Sınıf dengesi.** Çok dengesiz bir ayrım (ör. %95 aktif) modelin "her şeye
aktif" demesini ödüllendirir: doğruluk (accuracy) yüksek görünür ama model
hiçbir şey öğrenmemiştir. Dengeye yakın bir ayrım öğrenmeyi kolaylaştırır.

Aşağıdaki tablo, farklı eşiklerin veri setimizi nasıl böldüğünü gösterir. Kendi
çalışmanızda eşiği **önce farmakolojiye**, sonra dengeye bakarak seçin — ve
seçtiğiniz eşiği her zaman bildirin.

In [ ]:
satirlar = []
for esik in [5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0]:
    aktif = int((df[HEDEF_KOLON] >= esik).sum())
    satirlar.append({
        f"Esik ({HEDEF_KOLON})": esik,
        f"{AKTIVITE_TIPI} (nM)": f"{10 ** (9 - esik):,.0f}",
        "Aktif": aktif,
        "Inaktif": len(df) - aktif,
        "Aktif %": round(100 * aktif / len(df), 1),
        "Secildi": "<<<" if abs(esik - AKTIFLIK_ESIGI) < 1e-9 else "",
    })

df_esikler = pd.DataFrame(satirlar)
print(df_esikler.to_string(index=False))

n_aktif = int(df["Aktiflik"].sum())
print(f"\nSecilen esik: {HEDEF_KOLON} >= {AKTIFLIK_ESIGI}")
print(f"  Aktif   : {n_aktif:>6}  (%{100 * n_aktif / len(df):.1f})")
print(f"  Inaktif : {len(df) - n_aktif:>6}  (%{100 * (1 - n_aktif / len(df)):.1f})")

---
## 13. Görselleştirme

İki grafik çiziyoruz:

- **Solda** `pIC50` histogramı: dağılımın merkezi, yayılımı ve biçimi.
  Dağılım tek tepeli ve simetriğe yakın mı, çarpık mı, çift tepeli mi? Çift tepe,
  veri setinde farklı kimyasal serilerin ya da farklı deney koşullarının
  bulunduğuna işaret edebilir. Seçtiğimiz eşiği kesikli çizgiyle gösteriyoruz.
- **Sağda** sınıf dengesi.

Bu grafikler ileride model performansını yorumlarken referans noktasıdır:
verinin seyrek olduğu uç bölgelerde modelin daha zayıf tahmin yapması beklenir.

In [ ]:
fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.6))

# --- Sol: pIC50 dagilimi ---
sol.hist(df[HEDEF_KOLON], bins=40, color="#4878a8", edgecolor="white", alpha=0.9)
sol.axvline(AKTIFLIK_ESIGI, color="#c0392b", linestyle="--", linewidth=2,
            label=f"Esik = {AKTIFLIK_ESIGI}")
sol.axvline(df[HEDEF_KOLON].mean(), color="#2c3e50", linestyle=":", linewidth=2,
            label=f"Ortalama = {df[HEDEF_KOLON].mean():.2f}")
sol.set_xlabel(HEDEF_KOLON)
sol.set_ylabel("Molekul sayisi")
sol.set_title(f"{TARGET_ADI} — {HEDEF_KOLON} dagilimi  (n = {len(df)})")
sol.legend()
sol.grid(alpha=0.3, linestyle="--")

# --- Sag: sinif dengesi ---
sayim = df["Aktiflik"].value_counts().sort_index()
etiket = [f"Inaktif\n({HEDEF_KOLON} < {AKTIFLIK_ESIGI})",
          f"Aktif\n({HEDEF_KOLON} >= {AKTIFLIK_ESIGI})"]
cubuk = sag.bar(etiket, [sayim.get(0, 0), sayim.get(1, 0)],
                color=["#95a5a6", "#27ae60"], edgecolor="white")
for c, v in zip(cubuk, [sayim.get(0, 0), sayim.get(1, 0)]):
    sag.text(c.get_x() + c.get_width() / 2, v, f"{v}\n(%{100 * v / len(df):.1f})",
             ha="center", va="bottom", fontsize=10)
sag.set_ylabel("Molekul sayisi")
sag.set_title("Siniflandirma icin sinif dengesi")
sag.set_ylim(0, max(sayim) * 1.18)
sag.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.savefig(PROJE / f"ntrk1_veri_dagilimi.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafik kaydedildi: {PROJE / 'ntrk1_veri_dagilimi.png'}")

### En potent ve en zayıf moleküller

Sayılardan sonra **moleküllere bakmak** da önemlidir. Aşağıda en potent ve en
zayıf altı molekülü çiziyoruz.

Bakarken şunu sorun: en potent moleküller ortak bir iskelet (scaffold) paylaşıyor
mu? Genellikle paylaşırlar — çünkü bir ilaç keşif programı tek bir kimyasal seriyi
optimize eder. Bu gözlem ileride çok önemli olacak: rastgele bir eğitim/test
bölünmesi, aynı serinin moleküllerini hem eğitime hem teste dağıtır ve modelin
başarısını **olduğundan iyi** gösterir. Oturum 3'te bunun çözümü olan
**iskelet bazlı bölünmeyi** (scaffold split) göreceğiz.

In [ ]:
def molekul_izgarasi(alt_df, baslik, satir_sayisi=6):
    moller, etiketler = [], []
    for _, satir in alt_df.head(satir_sayisi).iterrows():
        mol = Chem.MolFromSmiles(satir["Smiles"])
        if mol is None:
            continue
        moller.append(mol)
        nm = 10 ** (9 - satir[HEDEF_KOLON])
        etiketler.append(f"{HEDEF_KOLON} {satir[HEDEF_KOLON]:.2f}  ({nm:,.1f} nM)")
    print(f"\n{baslik}")
    return Draw.MolsToGridImage(moller, legends=etiketler, molsPerRow=3,
                                subImgSize=(300, 230))


df_sirali = df.sort_values(HEDEF_KOLON, ascending=False)
molekul_izgarasi(df_sirali, "EN POTENT 6 MOLEKUL")

In [ ]:
molekul_izgarasi(df_sirali.iloc[::-1], "EN ZAYIF 6 MOLEKUL")

---
## 14. Veri setlerini kaydetme

Son adım: iki veri setini Drive'a yazıyoruz.

- **`ntrk1_regresyon.csv`** — iki kolon: `Smiles` (model girdisi) ve
  `pIC50` (model hedefi). Oturum 3'ün girdisi.
- **`ntrk1_siniflandirma.csv`** — üçüncü kolon olarak `Aktiflik` (0/1).
  Oturum 4'ün girdisi.

Yanına bir de **künye** yazıyoruz: hangi kaynaktan, hangi eşikle, hangi tarihte
üretildiği. Veri setinizin yanında böyle bir künye taşımak, aylar sonra "bu dosya
neydi?" sorusunu yanıtlamanın en kolay yoludur.

In [ ]:
from datetime import datetime

# Regresyon veri seti: Smiles -> pIC50
df[["Smiles", HEDEF_KOLON]].to_csv(YOL_REG, index=False)

# Siniflandirma veri seti: Smiles -> pIC50 + Aktiflik
df[["Smiles", HEDEF_KOLON, "Aktiflik"]].to_csv(YOL_SNF, index=False)

kunye_yerel = {
    "hedef": TARGET_ADI,
    "hedef_chembl_id": TARGET_CHEMBL_ID,
    "uniprot": "P04629",
    "aktivite_tipi": AKTIVITE_TIPI,
    "hedef_kolon": HEDEF_KOLON,
    "aktiflik_esigi": AKTIFLIK_ESIGI,
    "veri_kaynagi": KAYNAK,
    "uretim_zamani": datetime.now().isoformat(timespec="seconds"),
    "molekul_sayisi": int(len(df)),
    "aktif": int(df["Aktiflik"].sum()),
    "inaktif": int((1 - df["Aktiflik"]).sum()),
    f"{HEDEF_KOLON}_ortalama": round(float(df[HEDEF_KOLON].mean()), 3),
    f"{HEDEF_KOLON}_std": round(float(df[HEDEF_KOLON].std()), 3),
}

yol_kunye = PROJE / "ntrk1_oturum01_kunye.json"
yol_kunye.write_text(json.dumps(kunye_yerel, ensure_ascii=False, indent=2),
                     encoding="utf-8")

print("Kaydedilen dosyalar")
print("-" * 70)
for yol in (YOL_REG, YOL_SNF, yol_kunye):
    print(f"  {yol.name:<34} {yol.stat().st_size / 1024:>8.1f} KB")
print(f"\nKlasor: {PROJE}")

---
## 15. Özet

### Ne yaptık?

1. ChEMBL'e **dayanıklı** (yeniden denemeli) sorgu atan bir arayüz yazdık.
2. Hedefteki ölçüm tiplerini sayıp modelleme için **IC50**'yi seçtik.
3. Veriyi **iki yoldan** (canlı ve dondurulmuş) aldık ve karşılaştırdık.
4. Altı adımlı temizlik hattı uyguladık: eksik veri, tekrarlar, geçersiz yapılar
   ve **kanonikleştirme ile gizli tekrarlar**.
5. `pIC50` hedefini hesapladık ve sınıflandırma eşiğini gerekçeli
   biçimde seçtik.
6. Regresyon ve sınıflandırma veri setlerini Drive'a kaydettik.

### Akılda kalması gerekenler

- **Veri hazırlığı modelleme kadar önemlidir.** Sızıntılı ya da tutarsız bir veri
  seti, en iyi algoritmayı da işe yaramaz hâle getirir.
- **Kanonikleştirme bir lüks değil, zorunluluktur.** Gizli yapısal tekrarlar
  eğitim/test bölünmesini bozar.
- **Sansürlü değerleri (`>`, `<`) regresyona sokmayın.**
- **Birim tutarlılığı** olmadan pIC50 anlamsızdır.
- **Eşik bir bilimsel karardır:** farmakolojik anlam önce, sınıf dengesi sonra.
- **Veri sürümünü bildirin:** ChEMBL sürümü + indirme tarihi + filtreler.

### Sırada ne var?

**Oturum 2 — Özellik üretimi.** Elimizdeki `Smiles` kolonu hâlâ bir **metindir**;
model ise **sayılarla** öğrenir. Bir sonraki not defterinde her molekülü iki
tamamlayıcı yolla sayıya çevireceğiz: yorumlanabilir **moleküler betimleyiciler**
ve yapısal **parmak izleri**.

---

### Alıştırmalar

1. `AKTIFLIK_ESIGI`'ni 6.0 yapıp sınıf dengesinin nasıl değiştiğini gözlemleyin.
   Oturum 4'e geldiğinizde bu seçimin model metriklerini nasıl etkilediğini
   tartışacağız.
2. `TARGET_CHEMBL_ID`'yi `CHEMBL2599` (SYK kinazı) ya da `CHEMBL267` (SRC kinazı)
   olarak değiştirip not defterini baştan çalıştırın. Temizlik hunisi nasıl
   değişiyor? Hangi hedefin verisi daha "temiz"?
3. Adım 3'te molekül başına **minimum** değeri seçtik. Bunun yerine **medyan**
   almak daha mı iyi olurdu? Hangi durumda hangisi tercih edilir?
4. `standard_relation` filtresini kaldırıp (`>` ve `<` kayıtlarını da alarak)
   kaç molekülün eklendiğine bakın. Bu kayıtları modele katmanın bir yolu var mı?
   (İpucu: *censored regression*, *survival analysis*)